# 03 Evaluate Model

Load the best model checkpoint and evaluate it on the test split. Metrics are computed with `compute_bleu` and `compute_chrf` from `src/metrics.py`.

## Colab Setup

Run the next cell first when using Google Colab. It clones the repository, installs project dependencies from `pyproject.toml`, and moves the working directory to the repository root.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/songhahyun/seq2seq_transformer_model.git"
REPO_DIR = Path("/content/seq2seq_transformer_model")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", "dev", "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", ".[notebooks]"], check=True)
else:
    print("Not running in Colab; skipping clone/install.")

print(f"cwd: {Path.cwd()}")

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
from src.config import Config
from src.data_pipeline import (
    create_dataloaders,
    extract_pairs,
    load_ko_en_dataset,
    maybe_take_subset,
    set_seed,
    split_pairs,
)
from src.evaluate import generate_predictions, print_sample_translations
from src.metrics import compute_bleu, compute_chrf
from src.model_utils import load_checkpoint, load_model_from_checkpoint, load_tokenizers

config = Config()
set_seed(config.random_seed)

checkpoint_path = f"{config.checkpoint_dir}/best.pt"
print(f"device: {config.device}")
print(f"checkpoint: {checkpoint_path}")

## Load tokenizer and best checkpoint

In [ ]:
sp_src, sp_tgt = load_tokenizers(config)
checkpoint = load_checkpoint(checkpoint_path, config.device)
model = load_model_from_checkpoint(config, checkpoint, sp_src, sp_tgt)

print(f"checkpoint epoch: {checkpoint.get('epoch')}")
print(f"valid loss      : {checkpoint.get('valid_loss')}")
print(f"src vocab size  : {sp_src.get_piece_size()}")
print(f"tgt vocab size  : {sp_tgt.get_piece_size()}")

## Build test dataloader

In [ ]:
dataset = load_ko_en_dataset(
    config.dataset_name,
    split=config.train_split,
    hf_token=config.hf_token,
)
pairs = extract_pairs(dataset, src_col="ko", tgt_col="en")

train_pairs, valid_pairs, test_pairs = split_pairs(
    pairs,
    valid_ratio=config.valid_ratio,
    test_ratio=config.test_ratio,
    seed=config.random_seed,
)

train_pairs = maybe_take_subset(train_pairs, config.train_subset_size)
valid_pairs = maybe_take_subset(valid_pairs, config.valid_subset_size)
test_pairs = maybe_take_subset(test_pairs, config.test_subset_size)

_, _, test_loader = create_dataloaders(
    train_pairs,
    valid_pairs,
    test_pairs,
    sp_src,
    sp_tgt,
    config,
)

print(f"test pairs  : {len(test_pairs)}")
print(f"test batches: {len(test_loader)}")

## Generate predictions

In [ ]:
decode_results = {}
original_decode_strategy = config.decode_strategy
original_beam_size = config.beam_size

for decode_strategy in ("greedy", "beam"):
    config.decode_strategy = decode_strategy
    print(f"\n[Generate predictions: {decode_strategy}]")

    source_texts, predictions, references = generate_predictions(
        model=model,
        dataloader=test_loader,
        sp_tgt=sp_tgt,
        config=config,
        device=config.device,
    )

    decode_results[decode_strategy] = {
        "source_texts": source_texts,
        "predictions": predictions,
        "references": references,
        "beam_size": config.beam_size if decode_strategy == "beam" else 1,
    }

    print(f"predictions: {len(predictions)}")
    print(f"references : {len(references)}")

config.decode_strategy = original_decode_strategy
config.beam_size = original_beam_size

source_texts = decode_results["greedy"]["source_texts"]
predictions = decode_results["greedy"]["predictions"]
references = decode_results["greedy"]["references"]

## Compute BLEU and chrF

These functions are imported from `src/metrics.py`.

In [ ]:
metric_rows = []

for decode_strategy, result in decode_results.items():
    bleu = compute_bleu(result["predictions"], result["references"])
    chrf = compute_chrf(result["predictions"], result["references"])

    result["bleu"] = bleu
    result["chrf"] = chrf
    metric_rows.append({
        "decode_strategy": decode_strategy,
        "beam_size": result["beam_size"],
        "bleu": bleu,
        "chrf": chrf,
    })

    print(f"[{decode_strategy}] BLEU: {bleu:.4f}")
    print(f"[{decode_strategy}] chrF : {chrf:.4f}")

bleu = decode_results["greedy"]["bleu"]
chrf = decode_results["greedy"]["chrf"]

## Sample translations

In [ ]:
for decode_strategy, result in decode_results.items():
    print(f"\n[Sample Translations: {decode_strategy}]")
    print_sample_translations(
        source_texts=result["source_texts"],
        predictions=result["predictions"],
        references=result["references"],
        n=config.num_examples_for_samples,
    )